In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import time
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

# 1. Load the dataset
print("Loading dataset...")
df = pd.read_excel('Demand of last 2 years.xlsx', sheet_name='demand_training_data')

# 2. Data Preprocessing & Feature Engineering
print("Preprocessing and engineering features...")
df['week_start_date'] = pd.to_datetime(df['week_start_date'])
df['festival_event'] = df['festival_event'].fillna('None')

# Create the Lag Feature (Previous week's units sold)
df = df.sort_values(['location_id', 'product_id', 'week_start_date'])
df['prev_week_units_sold'] = df.groupby(['location_id', 'product_id'])['units_sold'].shift(1)
df = df.dropna(subset=['prev_week_units_sold'])

# Re-sort strictly by time for the train/test split
df = df.sort_values('week_start_date').reset_index(drop=True)

# Define target for classification (Stockout vs In-Stock)
target = 'stockout_flag'
features = [
    'year', 'week_number', 'location_id', 'region_of_kolkata',
    'product_id', 'category_name', 'brand', 'units_sold',
    'prev_week_units_sold', 'avg_selling_price_rs', 'promotion_flag',
    'discount_pct', 'holiday_flag', 'festival_event', 'season',
    'temperature_c', 'rainfall_mm', 'humidity_pct', 'weather_condition'
]

# Convert text variables to LightGBM categories
categorical_cols = [
    'location_id', 'region_of_kolkata', 'product_id', 'category_name',
    'brand', 'festival_event', 'season', 'weather_condition'
]
for col in categorical_cols:
    df[col] = df[col].astype('category')

# 3. Time-Series Train/Test Split (80/20)
split_index = int(len(df) * 0.8)

X_train = df.loc[:split_index, features]
y_train = df.loc[:split_index, target]

X_test = df.loc[split_index:, features]
y_test = df.loc[split_index:, target]

# 4. Train the LightGBM Classifier
print("Training LightGBM Classifier...")
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data, categorical_feature=categorical_cols)

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'verbose': -1,
    'random_state': 42
}

lgb_model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, test_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# 5. Make Predictions & Track Inference Speed
print("\nEvaluating model against holdout data...")
start_time = time.time()
y_pred_prob = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)
end_time = time.time()

# Threshold probabilities to get binary 0 or 1
y_pred = (y_pred_prob >= 0.5).astype(int)

inference_time_ms = (end_time - start_time) * 1000

# 6. Calculate Shootout Classification Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print(f"\n--- Classification Shootout Results ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Total Inference Time: {inference_time_ms:.2f} ms")
print(f"Confusion Matrix:\n{cm}")

Loading dataset...
Preprocessing and engineering features...
Training LightGBM Classifier...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[161]	training's binary_logloss: 0.0252622	valid_1's binary_logloss: 0.0932194

Evaluating model against holdout data...

--- Classification Shootout Results ---
Accuracy:  0.9782
Precision: 0.9231
Recall:    0.0952
F1 Score:  0.1727
Total Inference Time: 37.35 ms
Confusion Matrix:
[[5159    1]
 [ 114   12]]


In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load the dataset
print("Loading dataset...")
df = pd.read_excel('Demand of last 2 years.xlsx', sheet_name='demand_training_data')

# 2. Data Preprocessing
print("Preprocessing data...")
# Convert date columns to datetime objects
df['week_start_date'] = pd.to_datetime(df['week_start_date'])

# Fill missing values in 'festival_event' with 'None' since NaNs just mean no festival
df['festival_event'] = df['festival_event'].fillna('None')

# Select relevant features for the model
features = [
    'year', 'week_number', 'location_id', 'region_of_kolkata',
    'product_id', 'category_name', 'brand', 'units_sold',
    'avg_selling_price_rs', 'promotion_flag', 'discount_pct',
    'holiday_flag', 'festival_event', 'season', 'temperature_c',
    'rainfall_mm', 'humidity_pct', 'weather_condition', 'stockout_flag'
]

target = 'next_week_demand_target_units'

# Convert categorical columns to 'category' dtype for LightGBM
categorical_cols = [
    'location_id', 'region_of_kolkata', 'product_id', 'category_name',
    'brand', 'festival_event', 'season', 'weather_condition'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

# Sort the dataset chronologically (Critical for time-series forecasting)
df = df.sort_values('week_start_date').reset_index(drop=True)

# 3. Time-Series Train/Test Split
# Using the first 80% of chronological data for training and the last 20% for testing
split_index = int(len(df) * 0.8)

X_train = df.loc[:split_index, features]
y_train = df.loc[:split_index, target]

X_test = df.loc[split_index:, features]
y_test = df.loc[split_index:, target]

print(f"Training on {len(X_train)} samples, testing on {len(X_test)} samples.")

# 4. Train the LightGBM Model
print("Training LightGBM model...")

# Define LightGBM dataset format
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data, categorical_feature=categorical_cols)

# Set model hyperparameters for regression
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'verbose': -1,
    'random_state': 42
}

# Train the model with early stopping to prevent overfitting
lgb_model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, test_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)]
)

# 5. Make Predictions & Evaluate
print("Evaluating model...")
predictions = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"\n--- Model Performance ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} units")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} units")

# Optional: View the top 5 most important features
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values(by='Importance', ascending=False)

print("\n--- Top 5 Feature Importances ---")
print(importance_df.head())

Loading dataset...
Preprocessing data...
Training on 22241 samples, testing on 5560 samples.
Training LightGBM model...
Training until validation scores don't improve for 50 rounds
[50]	training's rmse: 11.8738	valid_1's rmse: 20.1744
[100]	training's rmse: 9.74138	valid_1's rmse: 20.3277
Early stopping, best iteration is:
[59]	training's rmse: 11.1202	valid_1's rmse: 19.9572
Evaluating model...

--- Model Performance ---
Mean Absolute Error (MAE): 14.45 units
Root Mean Squared Error (RMSE): 19.96 units

--- Top 5 Feature Importances ---
           Feature    Importance
7       units_sold  3.466339e+08
12  festival_event  4.994868e+07
1      week_number  1.180646e+07
13          season  1.013672e+07
14   temperature_c  7.311703e+06
